# Rooftop Solar — U-Net training on Kaggle GPU

Trains the repaired U-Net on the Swiss DOP25 set using the **leakage-free
geographic split** (F-05), with the corrected metric harness (F-03 / F-08).

Baseline to beat, measured locally with the fixed harness on the *old leaky*
split: **test IoU 0.517** (`path raise 130.pt`). Numbers here use the clean
geographic split, so they are stricter and not directly comparable — the honest
comparison is this run's test IoU against a local run on the same split.

Artifacts land in `/kaggle/working/runs/` and come back via `kaggle kernels output`.


In [ ]:
import subprocess, sys, os, json, time, shutil
from pathlib import Path

print(subprocess.run(["nvidia-smi","--query-gpu=name,memory.total,driver_version",
                      "--format=csv,noheader"], capture_output=True, text=True).stdout)
import torch
p = torch.cuda.get_device_properties(0)
print(f"torch {torch.__version__} | {p.name} | {p.total_memory/1024**3:.1f} GB | sm_{p.major}{p.minor}")
print("bf16 native:", p.major >= 8)

## 1. Get the code

The repo carries the 169 MB Swiss dataset, so no separate upload is needed.

In [ ]:
REPO = "https://github.com/Parthesh10/rooftop-solar-potential-detection.git"
WORK = Path("/kaggle/working")
SRC  = WORK / "repo"

if SRC.exists():
    shutil.rmtree(SRC)
subprocess.run(["git","clone","--depth","1",REPO,str(SRC)], check=True)
os.chdir(SRC)
sys.path.insert(0, str(SRC))
print("cloned at", subprocess.run(["git","rev-parse","--short","HEAD"],
      capture_output=True, text=True).stdout.strip())
print("train images:", len(list((SRC/"data"/"train"/"images").glob("*.png"))))

In [ ]:
# Kaggle images ship torch/numpy/opencv already. These are the extras.
subprocess.run([sys.executable,"-m","pip","install","-q","tifffile","nvidia-ml-py"],
               check=False)
import importlib
for m in ("numpy","cv2","PIL","tqdm","psutil"):
    try: importlib.import_module(m); print(f"  {m:<8} ok")
    except ImportError: print(f"  {m:<8} MISSING")

## 2. Regenerate the geographic split

Deterministic (`seed=0`), so this reproduces the exact manifests used locally:
420 train / 58 val / 74 test, with **zero** tiles adjacent across splits.

In [ ]:
r = subprocess.run([sys.executable,"-m","process_data.split",
    "--root","data","--out","data/splits",
    "--block-size","1000","--buffer","125"], capture_output=True, text=True)
print(r.stdout or r.stderr)

## 3. Train

Settings differ from the local 4 GB run: bigger batch, and AMP is left on `auto`
so `utils.select_amp` picks fp16 on a tensor-core card (T4/V100) and skips it on
a P100, after NaN-probing the real model.

The GPU governor is disabled here — Kaggle's cards are datacenter parts with
proper cooling, and throttling would only waste quota.

In [ ]:
RUNS = WORK / "runs"
RUNS.mkdir(exist_ok=True)
os.environ["RUNS_ROOT"] = str(RUNS)

cmd = [sys.executable,"-u","scripts/train_swiss.py",
       "--epochs","80", "--batch-size","16", "--lr","3e-4",
       "--workers","2", "--patience","15", "--amp","auto",
       "--gpu-util-target","100",   # no duty-cycling on datacenter GPUs
       "--gpu-temp-limit","0",      # no thermal guard
       "--gpu-mem-fraction","0.95",
       "--checkpoint-every","120",
       "--no-progress"]
print(" ".join(cmd), flush=True)

t0 = time.time()
proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                        text=True, bufsize=1)
for line in proc.stdout:
    print(line, end="", flush=True)
proc.wait()
print(f"
exit={proc.returncode}  elapsed={(time.time()-t0)/60:.1f} min")

## 4. Collect artifacts

In [ ]:
runs = sorted([d for d in RUNS.iterdir() if d.is_dir() and (d/"history.json").exists()],
              key=lambda d: d.stat().st_mtime)
if not runs:
    print("no completed run found")
else:
    run = runs[-1]
    print("run:", run.name)
    h = json.loads((run/"history.json").read_text())
    print(f"best val IoU {h['best_val_iou']:.4f} @ epoch {h['best_epoch']}"
          f" over {len(h['epochs'])} epochs")

    # Flatten what matters to the top of /kaggle/working so the output zip is small.
    for f in ("best.pt","history.json","metadata.json","train.log"):
        src = run/f
        if src.exists():
            shutil.copy2(src, WORK/f"{f}")
    # state.pt is large and only useful for resuming; drop it from the output.
    for d in RUNS.iterdir():
        for junk in ("state.pt","state.pt.bak","state.pt.tmp","last.pt"):
            (d/junk).unlink(missing_ok=True)
    shutil.rmtree(SRC, ignore_errors=True)   # don't ship the repo back
    print("
artifacts:", sorted(p.name for p in WORK.glob('*') if p.is_file()))

In [ ]:
# Curves, for a quick eyeball
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
if runs:
    fig, ax = plt.subplots(1,2, figsize=(13,4))
    ax[0].plot(h["train_loss"], label="train"); ax[0].plot(h["val_loss"], label="val")
    ax[0].set_title("loss"); ax[0].legend(); ax[0].grid(alpha=.3)
    ax[1].plot(h["train_iou"], label="train"); ax[1].plot(h["val_iou"], label="val")
    if h.get("best_epoch") is not None:
        ax[1].axvline(h["best_epoch"], ls="--", c="k", lw=.8)
    ax[1].set_title("IoU"); ax[1].legend(); ax[1].grid(alpha=.3)
    for a in ax: a.set_xlabel("epoch")
    plt.tight_layout(); plt.savefig(WORK/"curves.png", dpi=130); plt.show()